# T4: Storage format comparison

Benchmarks file size and read/write speed for Green taxi 2024 partition (617,885 rows).  
Compares Parquet, CSV, CSV.gz, and HDF5 in both Python and Snowflake environments.

In [ ]:
%%sql -r wh
USE WAREHOUSE BIGDATA_MZMB_WH

In [ ]:
import pyarrow
import h5py
print(f"pyarrow {pyarrow.__version__}, h5py {h5py.__version__}")

In [ ]:
import time
import os
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Loading Green 2024 partition...")
df = session.sql("""
    SELECT * FROM BIGDATA_TAXI_MZMB.SILVER.GREEN_TRIPS_CLEAN WHERE PICKUP_YEAR = 2024
""").to_pandas()
print(f"Loaded: {len(df):,} rows, {len(df.columns)} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
import time
import os
import numpy as np
import h5py

OUT_DIR = '/tmp/t4_formats'
os.makedirs(OUT_DIR, exist_ok=True)

results = []

def measure_write(name, write_fn):
    start = time.time()
    write_fn()
    return time.time() - start

def measure_read(name, read_fn):
    times = []
    for _ in range(3):
        start = time.time()
        _ = read_fn()
        times.append(time.time() - start)
    return np.median(times)

# --- Parquet (PyArrow) ---
parquet_path = f'{OUT_DIR}/green_2024.parquet'
w_time = measure_write('parquet', lambda: df.to_parquet(parquet_path, engine='pyarrow', compression='snappy'))
r_time = measure_read('parquet', lambda: pd.read_parquet(parquet_path, engine='pyarrow'))
size_mb = os.path.getsize(parquet_path) / 1e6
results.append(('Parquet (snappy)', size_mb, r_time, w_time, 'Python/PyArrow'))
print(f"Parquet: {size_mb:.1f} MB, read {r_time:.3f}s, write {w_time:.3f}s")

# --- CSV ---
csv_path = f'{OUT_DIR}/green_2024.csv'
w_time = measure_write('csv', lambda: df.to_csv(csv_path, index=False))
r_time = measure_read('csv', lambda: pd.read_csv(csv_path))
size_mb = os.path.getsize(csv_path) / 1e6
results.append(('CSV', size_mb, r_time, w_time, 'Python/pandas'))
print(f"CSV: {size_mb:.1f} MB, read {r_time:.3f}s, write {w_time:.3f}s")

# --- CSV gzip ---
csv_gz_path = f'{OUT_DIR}/green_2024.csv.gz'
w_time = measure_write('csv_gz', lambda: df.to_csv(csv_gz_path, index=False, compression='gzip'))
r_time = measure_read('csv_gz', lambda: pd.read_csv(csv_gz_path, compression='gzip'))
size_mb = os.path.getsize(csv_gz_path) / 1e6
results.append(('CSV (gzip)', size_mb, r_time, w_time, 'Python/pandas'))
print(f"CSV.gz: {size_mb:.1f} MB, read {r_time:.3f}s, write {w_time:.3f}s")

# --- HDF5 (h5py) ---
hdf_path = f'{OUT_DIR}/green_2024.h5'
def write_hdf5():
    with h5py.File(hdf_path, 'w') as f:
        for col in df.columns:
            data = df[col].values
            if data.dtype == object:
                data = data.astype(str)
                f.create_dataset(col, data=data.astype('S'))
            elif np.issubdtype(data.dtype, np.datetime64):
                f.create_dataset(col, data=data.astype('int64'))
            else:
                f.create_dataset(col, data=data, compression='gzip')

def read_hdf5():
    with h5py.File(hdf_path, 'r') as f:
        data = {col: f[col][:] for col in f.keys()}
    return pd.DataFrame(data)

w_time = measure_write('hdf5', write_hdf5)
r_time = measure_read('hdf5', read_hdf5)
size_mb = os.path.getsize(hdf_path) / 1e6
results.append(('HDF5 (h5py/gzip)', size_mb, r_time, w_time, 'Python/h5py'))
print(f"HDF5: {size_mb:.1f} MB, read {r_time:.3f}s, write {w_time:.3f}s")

print("\nAll benchmarks complete.")

In [ ]:
results_df = pd.DataFrame(results, columns=['FORMAT', 'FILE_SIZE_MB', 'READ_TIME_SEC', 'WRITE_TIME_SEC', 'ENGINE'])
results_df['ENGINE'] = 'Python'

session.sql('USE DATABASE BIGDATA_TAXI_MZMB').collect()
session.sql('USE WAREHOUSE BIGDATA_MZMB_WH').collect()
session.sql('CREATE OR REPLACE STAGE GOLD.T4_FORMAT_STAGE').collect()
session.sql("CREATE OR REPLACE FILE FORMAT GOLD.T4_PQ_FF TYPE='PARQUET'").collect()
session.sql("CREATE OR REPLACE FILE FORMAT GOLD.T4_CSV_FF TYPE='CSV' FIELD_OPTIONALLY_ENCLOSED_BY='\"' SKIP_HEADER=1").collect()
session.sql("CREATE OR REPLACE FILE FORMAT GOLD.T4_CSVGZ_FF TYPE='CSV' FIELD_OPTIONALLY_ENCLOSED_BY='\"' SKIP_HEADER=1 COMPRESSION='GZIP'").collect()

formats_sf = [
    ('Parquet (snappy)', 'parquet', "TYPE='PARQUET'", 'GOLD.T4_PQ_FF'),
    ('CSV', 'csv', "TYPE='CSV' COMPRESSION='NONE' FIELD_OPTIONALLY_ENCLOSED_BY='\"'", 'GOLD.T4_CSV_FF'),
    ('CSV (gzip)', 'csv_gz', "TYPE='CSV' COMPRESSION='GZIP' FIELD_OPTIONALLY_ENCLOSED_BY='\"'", 'GOLD.T4_CSVGZ_FF'),
]

def get_last_query_time():
    r = session.sql("""
        SELECT TOTAL_ELAPSED_TIME / 1000.0 AS SEC
        FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION(RESULT_LIMIT => 3))
        WHERE QUERY_TEXT NOT LIKE '%QUERY_HISTORY%'
        ORDER BY START_TIME DESC LIMIT 1
    """).to_pandas()
    return r['SEC'].iloc[0]

snowflake_results = []
for name, folder, write_ff, read_ff_name in formats_sf:
    session.sql(f"""
        COPY INTO @GOLD.T4_FORMAT_STAGE/{folder}/
        FROM (SELECT * FROM SILVER.GREEN_TRIPS_CLEAN WHERE PICKUP_YEAR = 2024)
        FILE_FORMAT = ({write_ff}) HEADER=TRUE OVERWRITE=TRUE
    """).collect()
    write_time = get_last_query_time()

    session.sql(f"""
        SELECT COUNT(*) FROM @GOLD.T4_FORMAT_STAGE/{folder}/ (FILE_FORMAT => '{read_ff_name}')
    """).collect()
    read_time = get_last_query_time()

    files = session.sql(f"LIST @GOLD.T4_FORMAT_STAGE/{folder}/").to_pandas()
    size_col = [c for c in files.columns if 'size' in c.lower()][0]
    size_mb = files[size_col].sum() / 1e6

    snowflake_results.append((name, size_mb, read_time, write_time, 'Snowflake'))
    print(f"  {name}: {size_mb:.1f} MB, read {read_time:.3f}s, write {write_time:.3f}s")

sf_df = pd.DataFrame(snowflake_results, columns=['FORMAT', 'FILE_SIZE_MB', 'READ_TIME_SEC', 'WRITE_TIME_SEC', 'ENGINE'])
combined = pd.concat([results_df, sf_df], ignore_index=True)
combined['LABEL'] = combined['FORMAT'] + ' [' + combined['ENGINE'] + ']'
print('\n' + combined[['LABEL', 'FILE_SIZE_MB', 'READ_TIME_SEC', 'WRITE_TIME_SEC']].to_string(index=False))

snow_df = session.create_dataframe(combined[['FORMAT', 'FILE_SIZE_MB', 'READ_TIME_SEC', 'WRITE_TIME_SEC', 'ENGINE']])
snow_df.write.mode('overwrite').save_as_table('BIGDATA_TAXI_MZMB.GOLD.T4_FORMAT_COMPARISON')
print('\nSaved to GOLD.T4_FORMAT_COMPARISON')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5.5))

plot_df = combined.sort_values('FILE_SIZE_MB', ascending=True)
colors = ['#2980B9' if e == 'Snowflake' else '#C0392B' for e in plot_df['ENGINE']]
ax1.barh(plot_df['LABEL'], plot_df['FILE_SIZE_MB'], color=colors, edgecolor='black', linewidth=0.4)
max_val = plot_df['FILE_SIZE_MB'].max()
for i, val in enumerate(plot_df['FILE_SIZE_MB']):
    if val > max_val * 0.7:
        ax1.text(val - max_val * 0.05, i, f'{val:.1f}', va='center', ha='right', fontsize=8, color='white', fontweight='bold')
    else:
        ax1.text(val + max_val * 0.01, i, f'{val:.1f}', va='center', fontsize=8)
ax1.set_xlabel('Size (MB)')
ax1.set_title('File size')
ax1.grid(True, alpha=0.2, axis='x')
ax1.set_xlim(0, max_val * 1.05)

plot_df = combined.sort_values('READ_TIME_SEC', ascending=True)
colors = ['#2980B9' if e == 'Snowflake' else '#C0392B' for e in plot_df['ENGINE']]
ax2.barh(plot_df['LABEL'], plot_df['READ_TIME_SEC'], color=colors, edgecolor='black', linewidth=0.4)
max_val = plot_df['READ_TIME_SEC'].max()
for i, val in enumerate(plot_df['READ_TIME_SEC']):
    if val > max_val * 0.7:
        ax2.text(val - max_val * 0.05, i, f'{val:.2f}s', va='center', ha='right', fontsize=8, color='white', fontweight='bold')
    else:
        ax2.text(val + max_val * 0.01, i, f'{val:.2f}s', va='center', fontsize=8)
ax2.set_xlabel('Time (seconds)')
ax2.set_title('Read speed (full scan)')
ax2.grid(True, alpha=0.2, axis='x')
ax2.set_xlim(0, max_val * 1.05)

plot_df = combined.sort_values('WRITE_TIME_SEC', ascending=True)
colors = ['#2980B9' if e == 'Snowflake' else '#C0392B' for e in plot_df['ENGINE']]
ax3.barh(plot_df['LABEL'], plot_df['WRITE_TIME_SEC'], color=colors, edgecolor='black', linewidth=0.4)
max_val = plot_df['WRITE_TIME_SEC'].max()
for i, val in enumerate(plot_df['WRITE_TIME_SEC']):
    if val > max_val * 0.7:
        ax3.text(val - max_val * 0.05, i, f'{val:.2f}s', va='center', ha='right', fontsize=8, color='white', fontweight='bold')
    else:
        ax3.text(val + max_val * 0.01, i, f'{val:.2f}s', va='center', fontsize=8)
ax3.set_xlabel('Time (seconds)')
ax3.set_title('Write speed')
ax3.grid(True, alpha=0.2, axis='x')
ax3.set_xlim(0, max_val * 1.05)

legend_elements = [Patch(facecolor='#2980B9', label='Snowflake'),
                   Patch(facecolor='#C0392B', label='Python')]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, 1.01))
plt.suptitle('Storage format benchmark - Green taxi 2024 (617,885 rows)', fontsize=12, y=1.05)
plt.tight_layout()
plt.show()